### Load and clean data

In [7]:
import pandas as pd

# Load data
df = pd.read_excel(
    r"C:\Users\Lenovo\Data_Analysis_Business\Project14_customer_analytics\data\data_online_retail.xlsx",
    engine="openpyxl"
)

# Basic checks
df.head()
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [8]:
# Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove returns (negative quantities)
df = df[df['Quantity'] > 0]

# Remove negative prices
df = df[df['UnitPrice'] > 0]

# Create Revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']

### Create variables

In [9]:
df['Month'] = df['InvoiceDate'].dt.to_period('M')

In [11]:
customer_summary = df.groupby('CustomerID').agg({
    'Revenue': 'sum',
    'InvoiceNo': 'nunique',
    'Quantity': 'sum'
}).reset_index()

customer_summary.columns = ['customer_id', 'total_revenue', 'num_orders', 'total_items']

# Average order value
customer_summary['avg_order_value'] = customer_summary['total_revenue'] / customer_summary['num_orders']

In [12]:
snapshot_date = df['InvoiceDate'].max()

recency_df = df.groupby('CustomerID')['InvoiceDate'].max().reset_index()
recency_df['recency_days'] = (snapshot_date - recency_df['InvoiceDate']).dt.days

customer_summary = customer_summary.merge(
    recency_df[['CustomerID', 'recency_days']],
    left_on='customer_id',
    right_on='CustomerID',
    how='left'
)

customer_summary.drop(columns=['CustomerID'], inplace=True)

In [13]:
customer_summary.head()
customer_summary.describe()

,customer_id,total_revenue,num_orders,total_items,avg_order_value,recency_days
count,4338.000000,4338.000000,4338.000000,4338.000000,4338.000000,4338.000000
mean,15300.408022,2054.266460,4.272015,1191.289073,419.166289,91.536422
std,1721.808492,8989.230441,7.697998,5046.081546,1796.537944,100.014169
min,12346.000000,3.750000,1.000000,1.000000,3.450000,0.000000
25%,13813.250000,307.415000,1.000000,160.000000,178.625000,17.000000
50%,15299.500000,674.485000,2.000000,379.000000,293.900000,50.000000
75%,16778.750000,1661.740000,5.000000,992.750000,430.113750,141.000000
max,18287.000000,280206.020000,209.000000,196915.000000,84236.250000,373.000000


### Save for Power BI

In [14]:
df.to_csv(r"C:\Users\Lenovo\Data_Analysis_Business\Project14_customer_analytics\data\transactions_clean.csv", index=False)
customer_summary.to_csv(r"C:\Users\Lenovo\Data_Analysis_Business\Project14_customer_analytics\data\customer_summary.csv", index=False)

### Create RFM Scores

In [15]:
# Recency score (LOW recency = GOOD)
customer_summary['R_score'] = pd.qcut(
    customer_summary['recency_days'],
    4,
    labels=[4,3,2,1]
)

# Frequency score (HIGH orders = GOOD)
customer_summary['F_score'] = pd.qcut(
    customer_summary['num_orders'].rank(method='first'),
    4,
    labels=[1,2,3,4]
)

# Monetary score (HIGH revenue = GOOD)
customer_summary['M_score'] = pd.qcut(
    customer_summary['total_revenue'],
    4,
    labels=[1,2,3,4]
)

In [16]:
customer_summary['RFM_score'] = (
    customer_summary['R_score'].astype(str) +
    customer_summary['F_score'].astype(str) +
    customer_summary['M_score'].astype(str)
)

In [17]:
def segment_customer(row):
    if row['R_score'] == 4 and row['F_score'] >= 3 and row['M_score'] >= 3:
        return 'High Value'
    elif row['R_score'] >= 3 and row['F_score'] >= 2:
        return 'Loyal'
    elif row['M_score'] == 4 and row['R_score'] <= 2:
        return 'At Risk High Value'
    elif row['F_score'] == 1:
        return 'One-Time Buyers'
    else:
        return 'Mid Value'

customer_summary['segment'] = customer_summary.apply(segment_customer, axis=1)

Check output

In [18]:
customer_summary['segment'].value_counts()

segment
Loyal                 1130
Mid Value             1113
One-Time Buyers       1064
High Value             799
At Risk High Value     232
Name: count, dtype: int64

Preview

In [19]:
customer_summary.head()

,customer_id,total_revenue,num_orders,total_items,avg_order_value,recency_days,R_score,F_score,M_score,RFM_score,segment
0,12346.0,77183.60,1,74215,77183.600000,325,1,1,4,114,At Risk High Value
1,12347.0,4310.00,7,2458,615.714286,1,4,4,4,444,High Value
2,12348.0,1797.24,4,2341,449.310000,74,2,3,4,234,At Risk High Value
3,12349.0,1757.55,1,631,1757.550000,18,3,1,4,314,One-Time Buyers
4,12350.0,334.40,1,197,334.400000,309,1,1,2,112,One-Time Buyers


### Save updated dataset

In [20]:
customer_summary.to_csv(
    r"C:\Users\Lenovo\Data_Analysis_Business\Project14_customer_analytics\data\customer_summary.csv",
    index=False
)